# Action Recommendation Notebook

This notebook focuses on operational action recommendations from delay risk scores.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [2]:
file_path = "/home/pratyush_device/Documents/college/AIML lab/Project/customer_cleaned.csv"
df = pd.read_csv(file_path)
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (750, 26)


,acquisition_cost_usd,order_value_usd,satisfaction_score,support_tickets,lead_time_days,days_acq_to_order,days_order_to_payment,acquisition_date_year,acquisition_date_month,acquisition_date_dayofweek,...,market_segment_Asia-Pacific,market_segment_Europe,market_segment_Middle East,market_segment_North America,supplier_id_SUP-B,supplier_id_SUP-C,supplier_id_SUP-D,supplier_id_SUP-E,supplier_id_SUP-F,supplier_id_SUP-G
0,850,12500,4,1,14,10,34,2024,1,4,...,False,False,False,True,False,False,False,False,False,False
1,920,8500,5,0,16,12,38,2024,1,5,...,False,True,False,False,True,False,False,False,False,False
2,880,21000,4,2,15,13,41,2024,1,6,...,True,False,False,False,False,True,False,False,False,False
3,790,7500,3,1,13,14,37,2024,1,0,...,False,False,False,True,False,False,False,False,False,False
4,950,15000,5,0,12,16,40,2024,1,1,...,False,True,False,False,False,False,True,False,False,False


In [3]:
target_col = "lead_time_days"
if target_col not in df.columns:
    raise ValueError("lead_time_days column not found in customer_cleaned.csv")

delay_threshold = df[target_col].quantile(0.75)
df["delay"] = (df[target_col] > delay_threshold).astype(int)

X_raw = df.drop(columns=[target_col, "delay"])
X = pd.get_dummies(X_raw, drop_first=True)
y = df["delay"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)

print("Delay threshold:", delay_threshold)
print("Model trained for recommendation scoring.")

Delay threshold: 16.0
Model trained for recommendation scoring.


In [4]:
def risk_level(prob):
    if prob >= 0.70:
        return "High"
    if prob >= 0.40:
        return "Medium"
    return "Low"


def action_plan(level):
    if level == "High":
        return {
            "Action": "Expedite shipment and escalate supplier",
            "Priority": "P1",
            "Owner": "Supply Chain Manager",
            "SLA_Hours": 12
        }
    if level == "Medium":
        return {
            "Action": "Monitor order and confirm dispatch timeline",
            "Priority": "P2",
            "Owner": "Operations Analyst",
            "SLA_Hours": 24
        }
    return {
        "Action": "No immediate action; keep routine monitoring",
        "Priority": "P3",
        "Owner": "Auto-monitoring",
        "SLA_Hours": 72
    }

In [5]:
# Build recommendations for all rows
delay_prob = clf.predict_proba(X)[:, 1]
recommendations = X_raw.copy()
recommendations["Delay_Probability"] = delay_prob
recommendations["Risk_Level"] = recommendations["Delay_Probability"].apply(risk_level)

action_df = recommendations["Risk_Level"].apply(action_plan).apply(pd.Series)
recommendations = pd.concat([recommendations, action_df], axis=1)

print(recommendations["Risk_Level"].value_counts())
recommendations.head(10)

Risk_Level
Low       658
High       90
Medium      2
Name: count, dtype: int64


,acquisition_cost_usd,order_value_usd,satisfaction_score,support_tickets,days_acq_to_order,days_order_to_payment,acquisition_date_year,acquisition_date_month,acquisition_date_dayofweek,order_date_year,...,supplier_id_SUP-D,supplier_id_SUP-E,supplier_id_SUP-F,supplier_id_SUP-G,Delay_Probability,Risk_Level,Action,Priority,Owner,SLA_Hours
0,850,12500,4,1,10,34,2024,1,4,2024,...,False,False,False,False,0.015,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
1,920,8500,5,0,12,38,2024,1,5,2024,...,False,False,False,False,0.280,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
2,880,21000,4,2,13,41,2024,1,6,2024,...,False,False,False,False,0.005,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
3,790,7500,3,1,14,37,2024,1,0,2024,...,False,False,False,False,0.140,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
4,950,15000,5,0,16,40,2024,1,1,2024,...,True,False,False,False,0.010,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
5,830,9500,4,1,18,42,2024,1,2,2024,...,False,True,False,False,0.055,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
6,910,18000,5,0,19,42,2024,1,3,2024,...,False,False,False,False,0.005,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
7,860,11000,4,0,21,42,2024,1,4,2024,...,False,False,True,False,0.015,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
8,900,13500,4,1,23,42,2024,1,5,2024,...,False,False,False,False,0.005,Low,No immediate action; keep routine monitoring,P3,Auto-monitoring,72
9,810,6500,3,2,25,41,2024,1,6,2024,...,False,False,False,True,0.910,High,Expedite shipment and escalate supplier,P1,Supply Chain Manager,12


In [6]:
# Action queue: show highest-risk orders first
queue_cols = ["Delay_Probability", "Risk_Level", "Priority", "Owner", "SLA_Hours", "Action"]
action_queue = recommendations.sort_values(by="Delay_Probability", ascending=False).reset_index(drop=True)
action_queue[queue_cols].head(-15)

,Delay_Probability,Risk_Level,Priority,Owner,SLA_Hours,Action
0,0.985,High,P1,Supply Chain Manager,12,Expedite shipment and escalate supplier
1,0.975,High,P1,Supply Chain Manager,12,Expedite shipment and escalate supplier
2,0.970,High,P1,Supply Chain Manager,12,Expedite shipment and escalate supplier
3,0.970,High,P1,Supply Chain Manager,12,Expedite shipment and escalate supplier
4,0.965,High,P1,Supply Chain Manager,12,Expedite shipment and escalate supplier
...,...,...,...,...,...,...
730,0.000,Low,P3,Auto-monitoring,72,No immediate action; keep routine monitoring
731,0.000,Low,P3,Auto-monitoring,72,No immediate action; keep routine monitoring
732,0.000,Low,P3,Auto-monitoring,72,No immediate action; keep routine monitoring
733,0.000,Low,P3,Auto-monitoring,72,No immediate action; keep routine monitoring


In [7]:
# Single-order recommendation helper
def recommend_for_order(input_row):
    input_encoded = pd.get_dummies(input_row, drop_first=True)
    input_encoded = input_encoded.reindex(columns=X.columns, fill_value=0)
    prob = float(clf.predict_proba(input_encoded)[0][1])
    level = risk_level(prob)
    plan = action_plan(level)
    return {
        "Delay_Probability": prob,
        "Risk_Level": level,
        **plan
    }

sample = X_raw.iloc[[0]]
recommend_for_order(sample)

{'Delay_Probability': 0.015,
 'Risk_Level': 'Low',
 'Action': 'No immediate action; keep routine monitoring',
 'Priority': 'P3',
 'Owner': 'Auto-monitoring',
 'SLA_Hours': 72}

In [8]:
output_path = "/home/pratyush_device/Documents/college/AIML lab/Project/action_recommendations.csv"
action_queue.to_csv(output_path, index=False)
print("Saved action recommendation file to:", output_path)

Saved action recommendation file to: /home/pratyush_device/Documents/college/AIML lab/Project/action_recommendations.csv
